## Global instructions

Only three input files are required (provided with example data):
1) AAseq.csv
2) overhangs.csv
3) orthogonal_oligos.csv

AAseq.csv should be your variant names and their associated amino acid sequences. Make sure all genes have unique names.

overhangs.csv defines the set of high-fiedlity Golden-gate overhangs to use. I use NEB GETset (https://ligasefidelity.neb.com/getset/run.cgi) and enter my required vector overhangs (GCTT and AGTG) to retrieve overhang sets compatible with those. I typically use 24 but more or less can also be used. 

orthogonal_oligos.csv contains the set or primers to use for amplification in 5'-3' orientation and will be used to add those binding sites to each sub-pool uniquely


In Gene Optimisation certain variables can be defined e.g. restriction sites to avoid, homopolymers to avoid, GC content, host to optimise for etc. all defined by the DNAchisel tool (https://edinburgh-genome-foundry.github.io/DnaChisel/index.html). This step can also be skipped if optimised elsewhere and supplied to Pool Assignment in the file Optimised_genes.csv

In Pool Assignment certain variables can be defined e.g. length of oPool (I typically use 300 or 250nt), the defined vector overhangs (to check they are not in your overhang list to assign to pools) and the maximum size of a 'short' sub-pool i.e. genes requiring only one fragment for assembly. 

In Primer and BsaI site addition certain variable can be defined e.g. Vector overhangs, the restriction enzyme site (including an A,G,C or T for its N e.g. BsaI is GGTCTCN| so I pass it GGTCTCA"). This will output three files, one FULL_INFO.csv with all information on pool assignment, primers to use, number of fragments etc., one references.fasta containing all gene sequences, and one oPool_Order_Fragments.csv which can be uploaded directly to Twist for ordering! 




## Gene optimisation

### Variables

In [ ]:
# Install deps once in your environment:


In [ ]:
import pandas as pd
from Bio.Seq import Seq
from Bio.Data import CodonTable
from dnachisel import (
    DnaOptimizationProblem,
    AvoidPattern,
    EnforceGCContent,
    EnforceTranslation,
    CodonOptimize,
)
# ─── User‐configurable parameters ───
# List of short sequences (e.g. restriction sites or other motifs) to avoid. Crucially include the restriction enzyme you will use for Golden-gate!
AVOID_SEQS = [
    "GGTCTC",      # BsaI
    #"CACCTGC", #PaqCI
    "AAAAA",      # Homopolymers
    "GGGGG", 
    "CCCCC",
    "TTTTT"     # hypothetical extra pattern
    # …add as many as you like…
]
# GC‐content constraints:
GC_MIN    = 0.30
GC_MAX    = 0.70
GC_WINDOW = 50

# Codon‐optimization settings:
CODON_SPECIES = "e_coli"
CODON_METHOD  = "match_codon_usage"
INPUT_PDB = "dTF001_dTF016"
input_path = "data/AAseq_dTF001_dTF016.csv"  # repo example input

output_path = "outputs/dTF001_dTF016_Optimised.csv"

input_df = pd.read_csv(input_path, names=["name", "aa_seq"])

input_df

### Script

#### Reverse translates amino acid sequences into DNA

In [ ]:
def reverse_translate(protein_sequence: str) -> str:
    """
    Reverse translate an amino acid sequence to a DNA sequence using the standard codon table.
    Chooses the first available codon for each amino acid.

    Args:
        protein_sequence (str): Amino acid sequence using 1-letter codes.

    Returns:
        str: A DNA sequence (string of A, T, G, C).
    """
    # Get the standard codon table
    codon_table = CodonTable.unambiguous_dna_by_id[1]

    # Create a dictionary mapping amino acids to one possible codon
    aa_to_codon = {}
    for codon, aa in codon_table.forward_table.items():
        if aa not in aa_to_codon:
            aa_to_codon[aa] = codon.upper()

    # Add stop codon
    aa_to_codon['*'] = codon_table.stop_codons[0].upper()

    # Translate the amino acid sequence
    dna_sequence = ''
    for aa in protein_sequence.upper():
        if aa not in aa_to_codon:
            raise ValueError(f"Invalid amino acid: {aa}")
        dna_sequence += aa_to_codon[aa]

    return dna_sequence


input_df["dna_seq_unrefined"] = input_df.aa_seq.apply(lambda x: reverse_translate(x))

#### Optimises DNA sequences

In [ ]:
def dna_chisel_remove_RS_and_codon_optimize(
    dna_seq: str,
    avoid_seqs: list[str] | None = None
) -> str:
    """
    Run DNA Chisel on `dna_seq`, removing any short motifs in `avoid_seqs`
    and then codon‐optimizing the entire open reading frame.

    If `avoid_seqs` is None, it uses the global AVOID_SEQS defined in Cell 1.
    """

    # Use the passed-in list or fall back to the global list
    patterns_to_avoid = avoid_seqs if (avoid_seqs is not None) else AVOID_SEQS

    seq_length = len(dna_seq)

    # Build one AvoidPattern constraint per motif in patterns_to_avoid
    avoid_constraints = [AvoidPattern(pat) for pat in patterns_to_avoid]

    problem = DnaOptimizationProblem(
        sequence=dna_seq,
        constraints=[
            *avoid_constraints,
            EnforceGCContent(mini=GC_MIN, maxi=GC_MAX, window=GC_WINDOW),
            EnforceTranslation(location=(0, seq_length)),
        ],
        objectives=[
            CodonOptimize(
                species=CODON_SPECIES,
                method=CODON_METHOD,
                location=(0, seq_length),
            )
        ]
    )

    # Solve and optimize
    problem.resolve_constraints()
    problem.optimize()

    return problem.sequence

input_df["dna_seq_optimized"] = input_df.dna_seq_unrefined.apply(lambda x: dna_chisel_remove_RS_and_codon_optimize(x))

input_df = input_df.drop(columns=["dna_seq_unrefined"])

input_df.to_csv(output_path, index=False)


## Pool Assignment

### Variables

In [ ]:
import os
import itertools
import pandas as pd
import networkx as nx
from pathlib import Path
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Data import CodonTable

# ----------------------------
# PARAMETERS / USER INPUT
# ----------------------------

# 1) oPool length options: 300, 250, 200, 150, 100
OPOOL_LENGTH = 300  # ← Change to one of [300, 250, 200, 150, 100]

# 2) Vector overhang modes:
#   - "fixed": use VECTOR_OVERHANG_1_FIXED as VectorOH1 for every gene
#   - "ser_windows": assign per-gene unique VectorOH1 from the 36-mer Ser-Ser window(0,2) set
VECTOR_OH1_MODE = "fixed"   # "fixed" or "ser_windows"
VECTOR_OVERHANG_1_FIXED = "GGCA"  # used only if VECTOR_OH1_MODE="fixed"

# Always-fixed VectorOH2 (must not appear as internal overhang)
VECTOR_OVERHANG_2 = "TCGG"

# 3) If VectorOH1 is assigned from Ser window0 subset (12-mers), Fragment 1 max is reduced by:
VEC1_WINDOW0_FRAG1_PENALTY_NT = 2

# 4) SHORT-pool limit - for one fragment assemblies only
SHORT_POOL_MAX_SIZE = None  # integer or None for “unlimited”

# 4.5) NEW: optionally strip N-terminal methionine from DNA if it starts with ATG
# If True, any gene whose dna_seq_optimized begins with "ATG" will have those first 3 nt removed.
STRIP_NTERM_MET = True

# Optional: write a log of which sequences were trimmed
WRITE_STRIP_LOG = True
STRIP_LOG_CSV = "outputs/dTF001_dTF016_stripped_ATG_log.csv"

# 5) Filepaths
OVERHANGS_FILE = "data/overhangs.csv"
INPUT_FILE     = output_path
OUTPUT_FILE    = "outputs/dTF001_dTF016_Assigned.csv"
UNASSIGNED_LOG = "outputs/dTF001_dTF016_unassigned.csv"


### Script

In [ ]:

# ----------------------------
# GLOBAL SETUP
# ----------------------------
def load_overhangs(path):
    """
    Reads a CSV whose first cell is a comma-separated list of 4-nt overhangs.
    Returns a list of 4-nt strings (uppercase).
    """
    df = pd.read_csv(path, header=None)
    raw = str(df.iloc[0, 0])
    return [oh.strip().upper() for oh in raw.split(",") if oh.strip()]

OVERHANGS_LIST = load_overhangs(OVERHANGS_FILE)

codon_table = CodonTable.unambiguous_dna_by_name["Standard"]
syn_map = {}
for codon, aa in codon_table.forward_table.items():
    syn_map.setdefault(aa, []).append(codon)

SER_CODONS = ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"]

# ----------------------------
# Ser-windows VectorOH1 sets (36 total = window0(12) ∪ window2(24))
# ----------------------------
def build_ser_windows_vec1_sets():
    w0 = set()
    w2 = set()
    for c1 in SER_CODONS:
        for c2 in SER_CODONS:
            s = c1 + c2
            w0.add(s[0:4])  # window0 = bases 0..3
            w2.add(s[2:6])  # window2 = bases 2..5
    union = w0 | w2
    return w0, w2, union

VEC1_W0_SET, VEC1_W2_SET, VEC1_SER_UNION = build_ser_windows_vec1_sets()

VECTOR_OH2 = VECTOR_OVERHANG_2.upper()
VECTOR_OH1_FIXED = VECTOR_OVERHANG_1_FIXED.upper()

# ----------------------------
# IO
# ----------------------------
def load_sequences(path):
    """
    Read a CSV with headers [name, aa_seq, dna_seq_optimized].
    Returns a list of SeqRecord, using 'name' as the record ID and
    'dna_seq_optimized' as the sequence.

    NEW:
      - If STRIP_NTERM_MET=True and sequence starts with ATG, strip the first 3 nt.
      - Collect a log of strip events for optional writing.
    """
    df = pd.read_csv(path)
    records = []
    strip_log = []  # list of dicts

    for _, row in df.iterrows():
        gid = str(row["name"]).strip()
        seq = str(row["dna_seq_optimized"]).strip().upper()
        if not (gid and seq):
            continue

        original_len = len(seq)
        stripped = False

        if STRIP_NTERM_MET:
            if seq.startswith("ATG"):
                seq = seq[3:]
                stripped = True

        if stripped:
            strip_log.append({
                "Sequence Name": gid,
                "Action": "Removed leading ATG",
                "Original_Length": original_len,
                "New_Length": len(seq),
                "Original_Start": "ATG",
                "New_Start": seq[:12] if len(seq) >= 12 else seq
            })

        records.append(SeqRecord(Seq(seq), id=gid, description=""))

    return records, strip_log


def write_unassigned_log(unassigned_ids, csv_path):
    pd.DataFrame({"Sequence Name": unassigned_ids}).to_csv(csv_path, index=False)


# ----------------------------
# Synonymous-edit machinery (unchanged behavior)
# ----------------------------
def try_synonymous_substitution(seq_str, codon_start):
    old_codon = seq_str[codon_start:codon_start + 3]
    aa = codon_table.forward_table.get(old_codon)
    if aa is None:
        return
    for syn_codon in syn_map.get(aa, []):
        if syn_codon == old_codon:
            continue
        yield seq_str[:codon_start] + syn_codon + seq_str[codon_start + 3:]


def enforce_overhang_at_pos(seq_str, overhang, start_pos):
    """
    Ensure seq_str[start_pos:start_pos+4] == overhang by ≤ one synonymous swap.
    Return (modified_seq, type_flag), where type_flag in {"Native","Synonymous"}.
    If impossible, return (None, None).
    """
    if seq_str[start_pos:start_pos + 4] == overhang:
        return seq_str, "Native"
    for offset in range(4):
        idx = start_pos + offset
        codon_start = idx - (idx % 3)
        if codon_start < 0 or codon_start + 3 > len(seq_str):
            continue
        for candidate in try_synonymous_substitution(seq_str, codon_start):
            if candidate[start_pos:start_pos + 4] == overhang:
                return candidate, "Synonymous"
    return None, None


def get_pos_to_options(seq_str, allowed_overhangs):
    """
    Build dict cut_pos -> list[(overhang, type_flag)] for allowed_overhangs only.
    """
    allowed = set([x.upper() for x in allowed_overhangs])
    L = len(seq_str)
    pos_to_options = {}
    codon_cache = {}

    for cut in range(4, L + 1):
        candidates = []

        window = seq_str[cut - 4:cut]
        if window in allowed:
            candidates.append((window, "Native"))

        for i in range(cut - 4, cut):
            codon_start = i - (i % 3)
            if codon_start < 0 or codon_start + 3 > L:
                continue
            if codon_start not in codon_cache:
                codon_cache[codon_start] = list(try_synonymous_substitution(seq_str, codon_start))
            for seq_mut in codon_cache[codon_start]:
                mut_window = seq_mut[cut - 4:cut]
                if mut_window in allowed:
                    candidates.append((mut_window, "Synonymous"))

        if candidates:
            seen = set()
            uniq = []
            for oh, tf in candidates:
                key = (oh, tf)
                if key not in seen:
                    seen.add(key)
                    uniq.append((oh, tf))
            pos_to_options[cut] = uniq

    return pos_to_options


# ----------------------------
# Fragment-length model (with optional W0 penalty on fragment 1)
# ----------------------------
def frag_max_len(fragment_index_1based, n_fragments, vec1_penalty_nt):
    if fragment_index_1based == n_fragments:
        return OPOOL_LENGTH - 62
    base = OPOOL_LENGTH - 58
    if fragment_index_1based == 1:
        base -= vec1_penalty_nt
    return base


def remaining_max_sum(after_cut_index, n_fragments, vec1_penalty_nt):
    rem_frags = list(range(after_cut_index + 2, n_fragments + 1))
    if not rem_frags:
        return 0
    rem_max = sum(frag_max_len(j, n_fragments, vec1_penalty_nt) for j in rem_frags)
    rem_cuts = (n_fragments - 1) - (after_cut_index + 1)
    return rem_max - 4 * rem_cuts


# ----------------------------
# Find internal overhangs + cuts for n-part assembly (uses pool-free internal OHs)
# ----------------------------
def find_valid_n_oh_combo(info, pool_free_internal_ohs, n_fragments, vec1_penalty_nt):
    seq_str = info["seq"]
    L = info["length"]
    pos_to_opts = info["pos_to_opts"]
    oh_needed = n_fragments - 1

    available_ohs = {oh for opts in pos_to_opts.values() for (oh, _) in opts}
    candidates = list(available_ohs & pool_free_internal_ohs)
    if len(candidates) < oh_needed:
        return None, None, None

    for oh_combo in itertools.permutations(sorted(candidates), oh_needed):
        seq_curr = seq_str
        prev_cut = 0
        cuts = []
        ok = True

        for cut_i, oh in enumerate(oh_combo):
            frag_idx = cut_i + 1
            frag_max = frag_max_len(frag_idx, n_fragments, vec1_penalty_nt)
            rem_max = remaining_max_sum(cut_i, n_fragments, vec1_penalty_nt)

            possible_cuts = sorted(
                [cut for cut, opts in pos_to_opts.items() if any(o == oh for (o, _) in opts)]
            )

            placed = False
            for cut in possible_cuts:
                if cut <= prev_cut:
                    continue
                frag_len = cut - prev_cut
                if frag_len > frag_max:
                    break

                rem_coding = L - (cut - 4)
                if rem_coding > rem_max:
                    continue

                seq_after, _ = enforce_overhang_at_pos(seq_curr, oh, cut - 4)
                if seq_after is None:
                    continue

                cuts.append(cut)
                seq_curr = seq_after
                prev_cut = cut
                placed = True
                break

            if not placed:
                ok = False
                break

        if ok:
            return list(oh_combo), seq_curr, cuts

    return None, None, None


# ----------------------------
# Pool builder for any n (supports fixed vec1 or ser_windows vec1)
# ----------------------------
def build_forbidden_internal_overhangs():
    forbidden = {VECTOR_OH2}
    if VECTOR_OH1_MODE == "fixed":
        forbidden.add(VECTOR_OH1_FIXED)
    elif VECTOR_OH1_MODE == "ser_windows":
        forbidden |= VEC1_SER_UNION
    else:
        raise ValueError("VECTOR_OH1_MODE must be 'fixed' or 'ser_windows'.")
    return forbidden


FORBIDDEN_INTERNAL = build_forbidden_internal_overhangs()
INTERNAL_OVERHANGS_LIST = [oh for oh in OVERHANGS_LIST if oh.upper() not in FORBIDDEN_INTERNAL]


def choose_vec1_from_free(free_vec1_set):
    free = set([x.upper() for x in free_vec1_set])

    w2 = sorted(list(free & VEC1_W2_SET))
    if w2:
        v = w2[0]
        return v, "W2", 0

    w0 = sorted(list(free & VEC1_W0_SET))
    if w0:
        v = w0[0]
        return v, "W0", VEC1_WINDOW0_FRAG1_PENALTY_NT

    return None, None, None


def max_bases_for_n(n_fragments, vec1_penalty_nt):
    total = 0
    for j in range(1, n_fragments + 1):
        total += frag_max_len(j, n_fragments, vec1_penalty_nt)
    total -= 4 * (n_fragments - 1)
    return total


def greedy_build_pools_for_n(genes, n_fragments, start_block_index):
    rows = []
    remaining = genes[:]
    next_block = start_block_index

    gene_info = {}
    for rec in remaining:
        seq_str = str(rec.seq)
        pos_to_opts = get_pos_to_options(seq_str, INTERNAL_OVERHANGS_LIST)
        gene_info[rec.id] = {"seq": seq_str, "length": len(seq_str), "pos_to_opts": pos_to_opts}

    def option_count(rec):
        info = gene_info[rec.id]
        return sum(len(opts) for opts in info["pos_to_opts"].values())

    remaining.sort(key=lambda r: (option_count(r), -len(r.seq), r.id))

    internal_capacity = len(INTERNAL_OVERHANGS_LIST) // (n_fragments - 1) if (n_fragments - 1) > 0 else 10**9
    if VECTOR_OH1_MODE == "ser_windows":
        vec1_capacity = len(VEC1_SER_UNION)
        pool_cap = min(internal_capacity, vec1_capacity)
    else:
        pool_cap = internal_capacity

    if pool_cap <= 0:
        return rows, remaining, next_block

    while remaining:
        pool_records = []
        free_internal = set(INTERNAL_OVERHANGS_LIST)
        free_vec1 = set(VEC1_SER_UNION) if VECTOR_OH1_MODE == "ser_windows" else set()

        i = 0
        while i < len(remaining) and len(pool_records) < pool_cap:
            rec = remaining[i]
            info = gene_info[rec.id]

            if not info["pos_to_opts"]:
                i += 1
                continue

            if VECTOR_OH1_MODE == "fixed":
                vec1, vec1_src, penalty = VECTOR_OH1_FIXED, "Fixed", 0
            else:
                vec1, vec1_src, penalty = choose_vec1_from_free(free_vec1)
                if vec1 is None:
                    break

            if info["length"] > max_bases_for_n(n_fragments, penalty):
                i += 1
                continue

            ohs, seq_mod, cuts = find_valid_n_oh_combo(info, free_internal, n_fragments, penalty)
            if ohs is None:
                i += 1
                continue

            pool_records.append((rec, vec1, vec1_src, penalty, ohs, cuts, seq_mod))
            for oh in ohs:
                free_internal.remove(oh)
            if VECTOR_OH1_MODE == "ser_windows":
                free_vec1.remove(vec1)

            remaining.pop(i)

        if not pool_records:
            break

        for (rec, vec1, vec1_src, penalty, ohs, cuts, seq_mod) in pool_records:
            full = seq_mod
            fragments = []
            prev = 0
            for cut in cuts:
                fragments.append(full[prev:cut])
                prev = cut - 4
            fragments.append(full[prev:])

            row = {
                "Block": next_block,
                "Length Distribution": f"Long-{n_fragments}part",
                "Sequence Name": rec.id,
                "VectorOH1": vec1,
                "VectorOH1_Source": vec1_src,
                "VectorOH2": VECTOR_OH2,
                "Full Sequence": full,
            }
            for k in range(n_fragments - 1):
                row[f"Overhang{k+1}"] = ohs[k] if k < len(ohs) else "N/A"
            for k in range(n_fragments):
                row[f"DNA Fragment {k+1}"] = fragments[k] if k < len(fragments) else ""
            rows.append(row)

        next_block += 1

    return rows, remaining, next_block


# ----------------------------
# Short phase
# ----------------------------
def assign_short_phase(short_recs, start_block):
    """
    Assign short sequences (single-fragment assemblies).

    FIX:
      - Fill VectorOH1 / VectorOH1_Source meaningfully instead of "N/A"
      - fixed mode: VectorOH1_FIXED, source="Fixed"
      - ser_windows mode: assign unique VectorOH1 per record within each short block
    """
    rows = []

    # Helper to assign vec1 for a block in ser_windows mode
    def vec1_generator():
        # deterministic order
        for v in sorted(VEC1_SER_UNION):
            # prefer W2 first (no penalty) then W0, consistent with your long logic
            if v in VEC1_W2_SET:
                yield (v, "W2")
        for v in sorted(VEC1_SER_UNION):
            if v in VEC1_W0_SET:
                yield (v, "W0")

    if SHORT_POOL_MAX_SIZE is None:
        # one "block" for all shorts
        if VECTOR_OH1_MODE == "fixed":
            for rec in short_recs:
                rows.append({
                    "Block": start_block,
                    "Length Distribution": "Short",
                    "Sequence Name": rec.id,
                    "VectorOH1": VECTOR_OH1_FIXED,
                    "VectorOH1_Source": "Fixed",
                    "VectorOH2": VECTOR_OH2,
                    "Overhang1": "N/A",
                    "Overhang2": "N/A",
                    "DNA Fragment 1": str(rec.seq),
                    "DNA Fragment 2": "",
                    "DNA Fragment 3": "",
                    "Full Sequence": str(rec.seq)
                })
        else:
            gen = vec1_generator()
            for rec in short_recs:
                try:
                    vec1, src = next(gen)
                except StopIteration:
                    raise ValueError(
                        f"Not enough unique ser-window VectorOH1 values for short pool. "
                        f"Need {len(short_recs)} but only {len(VEC1_SER_UNION)} available. "
                        f"Set SHORT_POOL_MAX_SIZE <= {len(VEC1_SER_UNION)} or use fixed mode."
                    )
                rows.append({
                    "Block": start_block,
                    "Length Distribution": "Short",
                    "Sequence Name": rec.id,
                    "VectorOH1": vec1,
                    "VectorOH1_Source": src,
                    "VectorOH2": VECTOR_OH2,
                    "Overhang1": "N/A",
                    "Overhang2": "N/A",
                    "DNA Fragment 1": str(rec.seq),
                    "DNA Fragment 2": "",
                    "DNA Fragment 3": "",
                    "Full Sequence": str(rec.seq)
                })

    else:
        # chunk shorts into blocks of size SHORT_POOL_MAX_SIZE
        block_idx = start_block
        for i in range(0, len(short_recs), SHORT_POOL_MAX_SIZE):
            chunk = short_recs[i:i + SHORT_POOL_MAX_SIZE]

            if VECTOR_OH1_MODE == "fixed":
                for rec in chunk:
                    rows.append({
                        "Block": block_idx,
                        "Length Distribution": "Short",
                        "Sequence Name": rec.id,
                        "VectorOH1": VECTOR_OH1_FIXED,
                        "VectorOH1_Source": "Fixed",
                        "VectorOH2": VECTOR_OH2,
                        "Overhang1": "N/A",
                        "Overhang2": "N/A",
                        "DNA Fragment 1": str(rec.seq),
                        "DNA Fragment 2": "",
                        "DNA Fragment 3": "",
                        "Full Sequence": str(rec.seq)
                    })
            else:
                gen = vec1_generator()
                if len(chunk) > len(VEC1_SER_UNION):
                    raise ValueError(
                        f"SHORT_POOL_MAX_SIZE={SHORT_POOL_MAX_SIZE} exceeds available unique "
                        f"ser-window VectorOH1 values ({len(VEC1_SER_UNION)}). "
                        f"Lower SHORT_POOL_MAX_SIZE or use fixed mode."
                    )
                for rec in chunk:
                    vec1, src = next(gen)
                    rows.append({
                        "Block": block_idx,
                        "Length Distribution": "Short",
                        "Sequence Name": rec.id,
                        "VectorOH1": vec1,
                        "VectorOH1_Source": src,
                        "VectorOH2": VECTOR_OH2,
                        "Overhang1": "N/A",
                        "Overhang2": "N/A",
                        "DNA Fragment 1": str(rec.seq),
                        "DNA Fragment 2": "",
                        "DNA Fragment 3": "",
                        "Full Sequence": str(rec.seq)
                    })

            block_idx += 1

    return rows



# ----------------------------
# MAIN PIPELINE
# ----------------------------
def main_pipeline():
    if VECTOR_OH1_MODE not in {"fixed", "ser_windows"}:
        raise ValueError("VECTOR_OH1_MODE must be 'fixed' or 'ser_windows'.")

    if VECTOR_OH2 in {x.upper() for x in OVERHANGS_LIST}:
        raise ValueError("Overhang file contains VECTOR_OVERHANG_2; internal selection forbids it, so remove it.")

    if VECTOR_OH1_MODE == "fixed" and VECTOR_OH1_FIXED in {x.upper() for x in OVERHANGS_LIST}:
        raise ValueError("Overhang file contains VECTOR_OVERHANG_1_FIXED; fixed vector mode expects it NOT in overhang file.")

    recs, strip_log = load_sequences(INPUT_FILE)

    if STRIP_NTERM_MET:
        print(f"[INFO] STRIP_NTERM_MET=True. Stripped leading ATG from {len(strip_log)} sequences (when present).")
        if WRITE_STRIP_LOG:
            Path(STRIP_LOG_CSV).parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(strip_log).to_csv(STRIP_LOG_CSV, index=False)
            print(f"[INFO] Wrote strip log -> {STRIP_LOG_CSV}")

    short_list, multi_list = [], []
    for rec in recs:
        if len(rec.seq) <= (OPOOL_LENGTH - 62):
            short_list.append(rec)
        else:
            multi_list.append(rec)

    rows = []
    short_rows = assign_short_phase(short_list, start_block=1)
    rows.extend(short_rows)
    current_block = (max(r["Block"] for r in short_rows) + 1) if short_rows else 1

    unassigned = []

    current_genes = multi_list[:]
    n = 2
    while current_genes:
        if (n - 1) > 0 and len(INTERNAL_OVERHANGS_LIST) < (n - 1):
            for rec in current_genes:
                unassigned.append(rec.id)
            break

        new_rows, remaining_genes, next_block = greedy_build_pools_for_n(
            current_genes, n_fragments=n, start_block_index=current_block
        )
        rows.extend(new_rows)
        current_block = next_block

        if len(remaining_genes) == len(current_genes):
            n += 1
            if n > 12:
                for rec in remaining_genes:
                    unassigned.append(rec.id)
                break
            current_genes = remaining_genes
        else:
            current_genes = remaining_genes

    df_out = pd.DataFrame(rows)
    if not df_out.empty:
        df_out.sort_values(by="Block", inplace=True)

        oh_cols = sorted([c for c in df_out.columns if c.startswith("Overhang")],
                         key=lambda x: int(x.replace("Overhang", "")))
        frag_cols = sorted([c for c in df_out.columns if c.startswith("DNA Fragment")],
                           key=lambda x: int(x.replace("DNA Fragment ", "")))

        base_cols = ["Block", "Length Distribution", "Sequence Name"]
        vec_cols = ["VectorOH1", "VectorOH1_Source", "VectorOH2"]

        final_order = base_cols + vec_cols + oh_cols + frag_cols + ["Full Sequence"]
        final_order = [c for c in final_order if c in df_out.columns]
        df_out = df_out[final_order]

    df_out.to_csv(OUTPUT_FILE, index=False)
    write_unassigned_log(unassigned, UNASSIGNED_LOG)

    print("Processing complete.")
    print(f"OPOOL_LENGTH         = {OPOOL_LENGTH}")
    print(f"STRIP_NTERM_MET      = {STRIP_NTERM_MET}")
    print(f"VECTOR_OH1_MODE      = {VECTOR_OH1_MODE}")
    if VECTOR_OH1_MODE == "fixed":
        print(f"VectorOH1 (fixed)    = {VECTOR_OH1_FIXED}")
    else:
        print(f"VectorOH1 pool size  = {len(VEC1_SER_UNION)} (W2={len(VEC1_W2_SET)} no-penalty, W0={len(VEC1_W0_SET)} penalty)")
        print(f"W0 penalty (nt)      = {VEC1_WINDOW0_FRAG1_PENALTY_NT}")
    print(f"VectorOH2 (fixed)    = {VECTOR_OH2}")
    print(f"Internal OH count    = {len(INTERNAL_OVERHANGS_LIST)} (forbids vector OHs)")
    print(f"Final CSV → {OUTPUT_FILE}")
    if unassigned:
        print(f"Unassigned → {UNASSIGNED_LOG} ({len(unassigned)} sequences)")
    else:
        print("All genes assigned.")


if __name__ == "__main__":
    main_pipeline()


## Primer assignment
Contains ability to use orthogonal oligos as unique primer pairs or combinatorially. Also handles compatibility with different vector overhang sources, for instance adding 2nt to finish of the 2 serine codons if using the approach for cell-free assembly

### Primer assignment with stuffer

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from Bio.Seq import Seq

# =========================
# USER CONFIG / VARIABLES
# =========================

# Target oligo length (final: FWD + (optional stuffer) + TypeIIS + insert(+vecOHs) + rc(TypeIIS) + rc(REV))
OPOOL_LENGTH = 300  # e.g. 300/250/200/150/100

# Optional stuffer to normalize all oligos to OPOOL_LENGTH
ADD_STUFFER = True
STUFFER_GC_MIN = 0.40
STUFFER_GC_MAX = 0.60
STUFFER_MAX_HOMOPOLYMER = 4
STUFFER_MAX_TRIES = 10000
STUFFER_SEED = 123  # set None for non-deterministic

# Type IIS site
TYPEIIS_CUT_SITE = "GGTCTCA"

# Primer assignment mode
#   - "unique_pairs": PRIMER_CSV is [F,R,F,R,...] and assigns one sequential pair per Block
#   - "combinatorial": PAIRS_CSV lists (Fwd,Rev) pairs; assigns one row per Block
PRIMER_ASSIGN_MODE = "unique_pairs"   # "unique_pairs" or "combinatorial"

# Unique-pairs mode settings
PRIMER_START_AT = None   # None/"" -> first primer; else match first column (numeric or string)
BLOCK_BASE      = 1

# Primer inventory
PRIMER_CSV            = "data/orthogonal_oligos.csv"
OUTPUT_UNUSED_PRIMERS = "outputs/orthogonal_oligos_unused.csv"
WRITE_UNUSED_PRIMERS  = True

# Combinatorial mode settings
GENERATE_PAIR_TABLE_ONLY = False
ALL_PAIRS_CSV            = "outputs/orthogonal_oligos_pairs_ALL.csv"
PAIRS_CSV                = "outputs/orthogonal_oligos_pairs_unused.csv"
OUTPUT_UNUSED_PAIRS      = "outputs/orthogonal_oligos_pairs_unused.csv"

# Assembly + outputs
INPUT_ASSEMBLY_CSV    = OUTPUT_FILE
OUTPUT_WITH_PRIMERS   = "outputs/dTF001_dTF016_FULL_INFO.csv"
OUTPUT_FASTA          = "outputs/dTF001_dTF016_references.fasta"
OUTPUT_FRAGMENTS_CSV  = "outputs/dTF001_dTF016_oPool_Order_Fragments.csv"

MAKE_OUTPUT_DIRS = True

# Vector overhang handling
# Assigned CSV may contain VectorOH1/VectorOH2/VectorOH1_Source; if missing or N/A we fall back.
FALLBACK_VECTOR_OH1 = "GGCA"
FALLBACK_VECTOR_OH2 = "TCGG"

# Which VectorOH1_Source labels require +2nt completion (your W0/window0 case)
VEC1_NEEDS_2NT_SOURCE_LABELS = {"W0"}


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from Bio.Seq import Seq

# =========================
# RNG
# =========================
_rng = np.random.default_rng(STUFFER_SEED) if (STUFFER_SEED is not None) else np.random.default_rng()

# =========================
# HELPERS
# =========================

def revcomp(s: str) -> str:
    return str(Seq(s).reverse_complement())

def is_nonempty_seq(x) -> bool:
    """True if x is a non-empty string (not NaN, not N/A)."""
    if x is None:
        return False
    if isinstance(x, float) and np.isnan(x):
        return False
    s = str(x).strip()
    return s != "" and s.upper() != "N/A"

def load_primers(path: str):
    df = pd.read_csv(path, header=None)
    df = df.replace(r'^\s*$', np.nan, regex=True).dropna(how='all').reset_index(drop=True)
    if df.shape[1] < 2:
        raise ValueError("Primer CSV must have ≥2 columns: first=Name/ID, last=Sequence.")
    name_col, seq_col = 0, df.shape[1] - 1
    names = df.iloc[:, name_col].astype(str).str.strip()
    seqs  = df.iloc[:, seq_col].astype(str).str.strip().str.upper()
    pairs = [(n, s) for n, s in zip(names, seqs) if n and s]
    if not pairs:
        raise ValueError("No valid (name, seq) rows parsed from primer CSV.")
    return pairs

def names_equal(a, b) -> bool:
    sa = str(a).strip(); sb = str(b).strip()
    try:
        return int(sa) == int(sb)
    except Exception:
        return sa == sb

def generate_all_pairs(primers, out_csv: str):
    rows = []
    n = len(primers)
    pair_id = 1
    for i in range(n):
        f_name, f_seq = primers[i]
        for j in range(i + 1, n):
            r_name, r_seq = primers[j]
            rows.append({
                "Pair_ID": pair_id,
                "Fwd_Name": f_name,
                "Fwd_Seq":  f_seq,
                "Rev_Name": r_name,
                "Rev_Seq":  r_seq,
            })
            pair_id += 1
    df_pairs = pd.DataFrame(rows)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df_pairs.to_csv(out_csv, index=False)
    print(f"[INFO] Generated {len(df_pairs)} unique primer pairs from {n} primers -> {out_csv}")

# =========================
# Ser-Ser completion logic (W0/window0 adds +2 nt)
# =========================
SER_CODONS = ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"]

def build_vec1_w0_suffix_map():
    """
    For SerSer = c1+c2 (6 nt), window0 4-mer is bases[0:4].
    If that 4-mer is used as VectorOH1, we need to add bases[4:6] (2 nt).
    Returns dict: vec1_4mer -> sorted list of possible 2nt suffixes.
    """
    m = {}
    for c1 in SER_CODONS:
        for c2 in SER_CODONS:
            s = c1 + c2
            vec1 = s[0:4]
            suf2 = s[4:6]
            m.setdefault(vec1, set()).add(suf2)
    return {k: sorted(list(v)) for k, v in m.items()}

VEC1_W0_SUFFIXES = build_vec1_w0_suffix_map()

def normalize_source_label(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip().upper()

def vec_overhangs_for_row(row):
    """
    Always returns usable (vec1, vec2, vec1_src):
      - if VectorOH1/2 are missing or N/A -> fall back
    """
    vec1 = row.get("VectorOH1", "")
    vec2 = row.get("VectorOH2", "")
    vec1_src = row.get("VectorOH1_Source", "")

    vec1 = str(vec1).strip().upper() if is_nonempty_seq(vec1) else ""
    vec2 = str(vec2).strip().upper() if is_nonempty_seq(vec2) else ""
    vec1_src = normalize_source_label(vec1_src)

    if not vec1:
        vec1 = FALLBACK_VECTOR_OH1.strip().upper()
    if not vec2:
        vec2 = FALLBACK_VECTOR_OH2.strip().upper()

    return vec1, vec2, vec1_src

def vec1_prefix_with_optional_2nt(vec1_4mer: str, vec1_source_label: str) -> str:
    """
    If vec1_source_label indicates W0/window0 case, return vec1 + extra2.
    Otherwise return vec1.
    """
    if vec1_source_label in {s.upper() for s in VEC1_NEEDS_2NT_SOURCE_LABELS}:
        opts = VEC1_W0_SUFFIXES.get(vec1_4mer, None)
        if not opts:
            raise ValueError(
                f"VectorOH1_Source='{vec1_source_label}' requires +2nt completion, "
                f"but vec1='{vec1_4mer}' is not a valid SerSer window0 4-mer."
            )
        extra2 = opts[0]  # deterministic
        return vec1_4mer + extra2
    return vec1_4mer

# =========================
# STUFFER GENERATION (ACGT only)
# =========================

def gc_fraction(seq: str) -> float:
    seq = seq.upper()
    gc = sum(1 for b in seq if b in ("G", "C"))
    return gc / len(seq) if seq else 0.0

def has_homopolymer_over(seq: str, max_run: int) -> bool:
    if not seq:
        return False
    run = 1
    prev = seq[0]
    for b in seq[1:]:
        if b == prev:
            run += 1
            if run > max_run:
                return True
        else:
            prev = b
            run = 1
    return False

def make_stuffer(length: int, forbidden_substrings: list) -> str:
    """
    Random A/C/G/T stuffer of exact length:
      - GC in [min,max] when feasible
      - no homopolymer run > max
      - does not contain forbidden substrings
      - NEVER contains N (hard-guarded)
    """
    if length <= 0:
        return ""

    forbidden_substrings = [s.upper() for s in forbidden_substrings if s]
    bases = np.array(list("ACGT"))  # <-- no N ever

    # Tiny lengths: GC band 0.40–0.60 is impossible for length 1 or 3.
    if length == 1:
        return "A"
    if length == 2:
        # exactly 50% GC -> one GC one AT
        for _ in range(STUFFER_MAX_TRIES):
            s_list = ["G", "A"]
            _rng.shuffle(s_list)
            s = "".join(s_list)
            if any(f in s for f in forbidden_substrings):
                continue
            return s
        return "GA"
    if length == 3:
        for s in ["ATG", "TAC", "GTA", "CAT"]:
            if all(f not in s for f in forbidden_substrings):
                return s
        return "ATG"
    if length == 4:
        # exactly 50% GC -> 2 GC + 2 AT
        for _ in range(STUFFER_MAX_TRIES):
            s_list = ["G", "C", "A", "T"]
            _rng.shuffle(s_list)
            s = "".join(s_list)
            if has_homopolymer_over(s, STUFFER_MAX_HOMOPOLYMER):
                continue
            if any(f in s for f in forbidden_substrings):
                continue
            return s
        return "GCAT"

    # General case (>=5)
    for _ in range(STUFFER_MAX_TRIES):
        s = "".join(_rng.choice(bases, size=length, replace=True))
        # hard guard (paranoia)
        if any(b not in "ACGT" for b in s):
            continue
        gcf = gc_fraction(s)
        if not (STUFFER_GC_MIN <= gcf <= STUFFER_GC_MAX):
            continue
        if has_homopolymer_over(s, STUFFER_MAX_HOMOPOLYMER):
            continue
        if any(f in s for f in forbidden_substrings):
            continue
        return s

    raise RuntimeError(
        f"Failed to generate valid stuffer of length {length} after {STUFFER_MAX_TRIES} tries."
    )

# =========================
# CORE ASSEMBLY
# =========================

def apply_pair_to_block_rows(df_block: pd.DataFrame,
                             fwd_name: str, fwd_seq: str,
                             rev_name: str, rev_seq: str,
                             frag_cols: list) -> pd.DataFrame:
    """
    For each row:
      - determine vec1/vec2 (row-specific; fallback if N/A/missing)
      - add vec1(+optional2nt) to FIRST non-empty fragment
      - add vec2 to LAST non-empty fragment (even if only one fragment -> both applied)
      - build oligo: FWD + stuffer(optional) + TypeIIS + (vecOHs+insert) + rc(TypeIIS) + rc(REV)
      - if ADD_STUFFER: make total length exactly OPOOL_LENGTH
    """
    cut_pref = TYPEIIS_CUT_SITE.strip().upper()
    cut_suf  = revcomp(cut_pref)
    rev_seq_suf = revcomp(rev_seq)

    forbid = [cut_pref, cut_suf]

    df_block = df_block.copy()

    for idx, row in df_block.iterrows():
        vec1, vec2, vec1_src = vec_overhangs_for_row(row)
        vec1_prefix = vec1_prefix_with_optional_2nt(vec1, vec1_src)

        non_empty_idx = [i for i, col in enumerate(frag_cols) if is_nonempty_seq(row.get(col, None))]

        new_frags = []
        for i, col in enumerate(frag_cols):
            seq = row.get(col, None)

            if not is_nonempty_seq(seq):
                # normalize missing to empty string
                new_frags.append("")
                continue

            seq = str(seq).strip().upper()

            first = (i == non_empty_idx[0]) if non_empty_idx else False
            last  = (i == non_empty_idx[-1]) if non_empty_idx else False

            s = seq
            if first:
                s = vec1_prefix + s
            if last:
                s = s + vec2

            core = cut_pref + s + cut_suf

            if ADD_STUFFER:
                fixed_len = len(fwd_seq) + len(core) + len(rev_seq_suf)
                stuffer_len = OPOOL_LENGTH - fixed_len
                if stuffer_len < 0:
                    raise ValueError(
                        f"Oligo would exceed OPOOL_LENGTH for row '{row.get('Sequence Name','?')}', "
                        f"Block={row.get('Block','?')}, fragment='{col}'. "
                        f"Need OPOOL_LENGTH≥{fixed_len} but OPOOL_LENGTH={OPOOL_LENGTH}."
                    )
                stuffer = make_stuffer(stuffer_len, forbidden_substrings=forbid)
            else:
                stuffer = ""

            oligo = fwd_seq + stuffer + core + rev_seq_suf

            if ADD_STUFFER and len(oligo) != OPOOL_LENGTH:
                raise AssertionError(f"Built oligo length {len(oligo)} != OPOOL_LENGTH {OPOOL_LENGTH}")
            if any(b not in "ACGT" for b in oligo):
                raise AssertionError("Found non-ACGT base in oligo (stuffer must be A/C/G/T only).")

            new_frags.append(oligo)

        for col, nf in zip(frag_cols, new_frags):
            df_block.at[idx, col] = nf

        df_block.at[idx, "Primer_Forward_Name"] = fwd_name
        df_block.at[idx, "Primer_Forward_Seq"]  = fwd_seq
        df_block.at[idx, "Primer_Reverse_Name"] = rev_name
        df_block.at[idx, "Primer_Reverse_Seq"]  = rev_seq

    return df_block

def ensure_output_dirs():
    if not MAKE_OUTPUT_DIRS:
        return
    for p in [OUTPUT_WITH_PRIMERS, OUTPUT_FASTA, OUTPUT_FRAGMENTS_CSV,
              OUTPUT_UNUSED_PRIMERS, ALL_PAIRS_CSV, OUTPUT_UNUSED_PAIRS]:
        if p:
            Path(p).parent.mkdir(parents=True, exist_ok=True)

# =========================
# MAIN
# =========================

ensure_output_dirs()

df = pd.read_csv(INPUT_ASSEMBLY_CSV)

if "Block" not in df.columns:
    raise ValueError("Input assembly CSV must contain a 'Block' column.")
df["Block"] = pd.to_numeric(df["Block"], errors="raise").astype(int)

frag_cols = [c for c in df.columns if c.startswith("DNA Fragment")]
if not frag_cols:
    raise ValueError("No 'DNA Fragment' columns found in INPUT_ASSEMBLY_CSV.")
frag_cols_sorted = sorted(frag_cols, key=lambda x: int(x.replace("DNA Fragment ", "")))

unique_blocks = sorted(df["Block"].unique())
num_blocks = len(unique_blocks)

if "VectorOH1" not in df.columns:
    print("[WARN] 'VectorOH1' column not found in assigned CSV. Falling back to FALLBACK_VECTOR_OH1.")
if "VectorOH2" not in df.columns:
    print("[WARN] 'VectorOH2' column not found in assigned CSV. Falling back to FALLBACK_VECTOR_OH2.")
if "VectorOH1_Source" not in df.columns:
    print("[INFO] 'VectorOH1_Source' not found — will NOT add +2nt SerSer completion unless present.")

primers = load_primers(PRIMER_CSV)

if PRIMER_ASSIGN_MODE == "combinatorial" and GENERATE_PAIR_TABLE_ONLY:
    generate_all_pairs(primers, ALL_PAIRS_CSV)
    print("[DONE] Pair table generation only.")
    raise SystemExit(0)

# =========================
# MODE 1: UNIQUE PAIRS
# =========================
if PRIMER_ASSIGN_MODE == "unique_pairs":
    if PRIMER_START_AT is None or (isinstance(PRIMER_START_AT, str) and PRIMER_START_AT.strip() == ""):
        start_idx = 0
        print(f"[INFO] PRIMER_START_AT not specified — using FIRST primer in {PRIMER_CSV} as forward primer for Block={BLOCK_BASE}.")
    else:
        start_idx = next((i for i, (nm, _) in enumerate(primers) if names_equal(nm, PRIMER_START_AT)), None)
        if start_idx is None:
            raise ValueError(
                f"Primer named/ID '{PRIMER_START_AT}' not found in {PRIMER_CSV} (first column). "
                "Set PRIMER_START_AT=None to start from the first primer."
            )
        print(f"[INFO] Using primer '{primers[start_idx][0]}' as the explicit start primer for Block={BLOCK_BASE}.")

    if start_idx % 2 == 1:
        print(
            f"[WARN] Start primer '{primers[start_idx][0]}' is at odd index {start_idx} (0-based). "
            f"If your file is [F,R,F,R,...], this may be a reverse. Using it as FORWARD as requested."
        )

    def primers_for_block(block_val: int):
        k = block_val - BLOCK_BASE
        if k < 0:
            raise ValueError(f"Block {block_val} < BLOCK_BASE {BLOCK_BASE}; adjust BLOCK_BASE or your data.")
        fwd_i = start_idx + 2 * k
        rev_i = fwd_i + 1
        if rev_i >= len(primers):
            raise IndexError(
                f"Not enough primers for Block {block_val}: need indices up to {rev_i} (0-based), "
                f"but only {len(primers)} primers available."
            )
        (fwd_name, fwd_seq) = primers[fwd_i]
        (rev_name, rev_seq) = primers[rev_i]
        return fwd_name, fwd_seq, rev_name, rev_seq

    df_out_blocks = []
    for blk in unique_blocks:
        fwd_name, fwd_seq, rev_name, rev_seq = primers_for_block(int(blk))
        df_block = df[df["Block"] == blk]
        df_block = apply_pair_to_block_rows(df_block, fwd_name, fwd_seq, rev_name, rev_seq, frag_cols_sorted)
        df_out_blocks.append(df_block)

    df_out = pd.concat(df_out_blocks, axis=0, ignore_index=True).sort_values(by="Block")

    if WRITE_UNUSED_PRIMERS:
        used_names = set(df_out["Primer_Forward_Name"].astype(str).str.strip()) | \
                     set(df_out["Primer_Reverse_Name"].astype(str).str.strip())
        unused_pairs = [(name, seq) for (name, seq) in primers if str(name).strip() not in used_names]

        if unused_pairs:
            pd.DataFrame(unused_pairs, columns=["Primer_Name", "Sequence"]).to_csv(
                OUTPUT_UNUSED_PRIMERS, index=False, header=False
            )
            print(f"[INFO] Wrote unused primers → {OUTPUT_UNUSED_PRIMERS} ({len(unused_pairs)} of {len(primers)} remain; no header).")
        else:
            print("[INFO] All primers used; no unused primers to write.")

# =========================
# MODE 2: COMBINATORIAL
# =========================
elif PRIMER_ASSIGN_MODE == "combinatorial":
    pairs_df = pd.read_csv(PAIRS_CSV)

    required_pair_cols = {"Fwd_Name", "Fwd_Seq", "Rev_Name", "Rev_Seq"}
    if not required_pair_cols.issubset(set(pairs_df.columns)):
        raise ValueError(f"Pairs CSV must contain columns {required_pair_cols}. Found: {list(pairs_df.columns)}")

    if len(pairs_df) < num_blocks:
        raise ValueError(f"Not enough primer pairs: {len(pairs_df)} pairs available but {num_blocks} blocks present.")

    print(f"[INFO] Assigning {num_blocks} primer pairs to {num_blocks} blocks using first {num_blocks} rows from {PAIRS_CSV}.")

    df_out_blocks = []
    for i, blk in enumerate(unique_blocks):
        pr = pairs_df.iloc[i]
        fwd_name = str(pr["Fwd_Name"]).strip()
        fwd_seq  = str(pr["Fwd_Seq"]).strip().upper()
        rev_name = str(pr["Rev_Name"]).strip()
        rev_seq  = str(pr["Rev_Seq"]).strip().upper()

        df_block = df[df["Block"] == blk]
        df_block = apply_pair_to_block_rows(df_block, fwd_name, fwd_seq, rev_name, rev_seq, frag_cols_sorted)
        df_out_blocks.append(df_block)

    df_out = pd.concat(df_out_blocks, axis=0, ignore_index=True).sort_values(by="Block")

    if len(pairs_df) > num_blocks:
        unused_pairs_df = pairs_df.iloc[num_blocks:].reset_index(drop=True)
        unused_pairs_df.to_csv(OUTPUT_UNUSED_PAIRS, index=False)
        print(f"[INFO] Wrote unused pairs -> {OUTPUT_UNUSED_PAIRS} ({len(unused_pairs_df)} of {len(pairs_df)} remain).")
    else:
        print("[INFO] All available pairs used; no unused pairs file written.")

else:
    raise ValueError("PRIMER_ASSIGN_MODE must be 'unique_pairs' or 'combinatorial'.")

# =========================
# OUTPUTS
# =========================

oh_cols = sorted([c for c in df_out.columns if c.startswith("Overhang")],
                 key=lambda x: int(x.replace("Overhang", ""))) if any(c.startswith("Overhang") for c in df_out.columns) else []

base_cols   = ["Block", "Length Distribution", "Sequence Name"]
primer_cols = ["Primer_Forward_Name", "Primer_Forward_Seq", "Primer_Reverse_Name", "Primer_Reverse_Seq"]
vec_cols = [c for c in ["VectorOH1", "VectorOH1_Source", "VectorOH2"] if c in df_out.columns]

col_order = base_cols + vec_cols + oh_cols + frag_cols_sorted
if "Full Sequence" in df_out.columns:
    col_order += ["Full Sequence"]
col_order += primer_cols

for c in col_order:
    if c not in df_out.columns:
        df_out[c] = ""
df_out = df_out[col_order]

df_out.to_csv(OUTPUT_WITH_PRIMERS, index=False)

if "Full Sequence" in df_out.columns:
    with open(OUTPUT_FASTA, "w") as fh:
        for _, row in df_out.iterrows():
            fh.write(f">Block_{int(row['Block'])}_{row['Sequence Name']}\n{row['Full Sequence']}\n")
    print(f"[INFO] Wrote FASTA -> {OUTPUT_FASTA}")

frag_rows = []
for _, row in df_out.iterrows():
    seq_name = row["Sequence Name"]
    for i, col in enumerate(frag_cols_sorted, start=1):
        frag = row[col]
        if is_nonempty_seq(frag):
            frag_rows.append({"Gene_Fragment": f"{seq_name}_Fragment{i}", "Sequence": str(frag)})
pd.DataFrame(frag_rows).to_csv(OUTPUT_FRAGMENTS_CSV, index=False)

print(f"[INFO] Wrote FULL_INFO -> {OUTPUT_WITH_PRIMERS}")
print(f"[INFO] Wrote fragments CSV -> {OUTPUT_FRAGMENTS_CSV}")
print("[DONE] Primer assignment complete.")


## Merging oPool order fragment csvs

In [ ]:
import pandas as pd
from pathlib import Path

# Root directory containing all opTF001 subfolders
ROOT_DIR = Path("outputs")

# Output CSV
OUT_CSV = ROOT_DIR / "opTF003_oPool_Order_Fragments.csv"

# Find all CSVs whose names end with "oPool_Order_Fragments.csv"
csv_files = sorted(ROOT_DIR.rglob("*oPool_Order_Fragments.csv"))

print("Found files:")
for f in csv_files:
    print(" -", f)

if not csv_files:
    raise FileNotFoundError("No *oPool_Order_Fragments.csv files found under opTF001")

# Read and merge all files (they already share the same header)
dfs = [pd.read_csv(f) for f in csv_files]
merged = pd.concat(dfs, ignore_index=True)

# Write out a single CSV with one header row
merged.to_csv(OUT_CSV, index=False)

print(f"\nSaved merged CSV to:\n{OUT_CSV}")


## Random crap

In [ ]:
from __future__ import annotations

import re
import math
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Sequence, Tuple, Optional

import numpy as np
import pandas as pd

# ============================================================
# PART 1 — PASTE YOUR TABLE HERE
# ============================================================
RAW_TABLE = r""" 	AGTG	CACT	TTCT	AGAA	TTCC	GGAA	TTCA	TGAA	TTCG	CGAA	CTCT	AGAG	CTCC	GGAG	CTCA	TGAG	CTCG	CGAG	ATCT	AGAT	ATCC	GGAT	ATCA	TGAT	ATCG	CGAT	GTCT	AGAC	GTCC	GGAC	GTCA	TGAC	GTCG	CGAC	TAGT	ACTA	TAGC	GCTA	CAGT	ACTG	CAGC	GCTG	AAGT	ACTT	AAGC	GCTT	GAGT	ACTC	GAGC	GCTC	GGTA	TACC	AAAT	ATTT	AAGA	TCTT	AGGA	TCCT	CCCC	GGGG	GTTA	TAAC	ACCA	TGGT	CTAA	TTAG	GTGA	TCAC	AATC	GATT	CATA	TATG	TACA	TGTA	ACGA	TCGT	ATAA	TTAT	TAAA	TTTA	TCAA	TTGA	AAAG	CTTT	AATA	TATT	TAGA	TCTA	GAAA	TTTC	CCGA	TCGG	CGGA	TCCG	CCAG	CTGG	ATGG	CCAT	AGCA	TGCT
AGTG	0	191	0	0	0	0	0	0	0	0	22	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1
CACT	191	1	0	0	0	0	0	0	0	0	0	8	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TTCT	0	0	0	368	0	16	0	14	0	1	0	29	0	1	0	1	0	0	0	9	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	11	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0
AGAA	0	0	368	0	3	0	4	0	3	0	5	0	0	0	0	0	0	0	13	0	0	0	0	0	0	0	7	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1
TTCC	0	0	0	3	0	351	0	1	0	0	0	0	0	19	0	0	0	0	0	0	0	18	0	0	0	0	0	0	0	5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0
GGAA	0	0	16	0	351	0	2	0	1	0	0	0	11	0	0	0	0	0	0	0	19	0	0	0	0	0	0	0	7	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	10	0	0	0	0	0	0	0	0	0	0
TTCA	0	0	0	4	0	2	0	265	0	2	0	0	0	0	0	11	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TGAA	0	0	14	0	1	0	265	0	14	0	0	0	0	0	3	0	0	0	0	0	0	0	13	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TTCG	0	0	0	3	0	1	0	14	0	322	0	0	0	0	0	0	0	43	0	0	0	0	0	0	0	21	0	0	0	0	0	0	0	5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	24	0	0	0	0	0	0	0
CGAA	0	0	1	0	0	0	2	0	322	0	0	0	0	0	0	0	8	0	0	0	0	0	0	0	25	0	0	0	0	0	0	0	18	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0
CTCT	22	0	0	5	0	0	0	0	0	0	0	277	0	31	0	31	0	16	0	2	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AGAG	0	8	29	0	0	0	0	0	0	0	277	0	19	0	23	0	6	0	5	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CTCC	0	0	0	0	0	11	0	0	0	0	0	19	0	195	0	10	0	11	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	11	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GGAG	0	0	1	0	19	0	0	0	0	0	31	0	195	0	21	0	2	0	0	0	7	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CTCA	0	0	0	0	0	0	0	3	0	0	0	23	0	21	0	234	0	31	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TGAG	0	0	1	0	0	0	11	0	0	0	31	0	10	0	234	0	14	0	0	0	0	0	5	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CTCG	0	0	0	0	0	0	0	0	0	8	0	6	0	2	0	14	0	225	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0
CGAG	0	0	0	0	0	0	0	0	43	0	16	0	11	0	31	0	225	1	0	0	0	0	0	0	22	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0
ATCT	0	0	0	13	0	0	0	0	0	0	0	5	0	0	0	0	0	0	0	319	0	24	0	27	0	6	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AGAT	0	0	9	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	319	0	9	0	21	0	21	0	33	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	12	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ATCC	0	0	0	0	0	19	0	0	0	0	0	0	0	7	0	0	0	0	0	9	0	284	0	5	0	8	0	0	0	22	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GGAT	0	0	0	0	18	0	0	0	0	0	0	0	4	0	0	0	0	0	24	0	284	0	11	0	3	0	0	0	31	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ATCA	0	0	0	0	0	0	0	13	0	0	0	0	0	0	0	5	0	0	0	21	0	11	0	271	0	12	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	7	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6
TGAT	0	0	0	0	0	0	3	0	0	0	0	0	0	0	1	0	0	0	27	0	5	0	271	0	38	0	1	0	0	0	28	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	10	0
ATCG	0	0	0	0	0	0	0	0	0	25	0	0	0	0	0	0	0	22	0	21	0	3	0	38	1	280	0	0	0	0	0	0	0	22	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0
CGAT	0	0	0	0	0	0	0	0	21	0	0	0	0	0	0	0	2	0	6	0	8	0	12	0	280	0	0	0	0	0	0	0	37	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	5	0	0	0
GTCT	0	0	0	7	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	33	0	0	0	1	0	0	0	243	0	33	0	43	0	22	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AGAC	0	0	1	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	10	0	0	0	0	0	0	0	243	0	21	0	46	0	25	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GTCC	0	0	0	0	0	7	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	31	0	0	0	0	0	21	0	263	0	18	0	19	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GGAC	0	0	0	0	5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	22	0	0	0	0	0	33	0	263	1	28	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GTCA	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	28	0	0	0	46	0	28	0	276	0	50	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TGAC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	43	0	18	0	276	0	32	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	16	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GTCG	0	0	0	0	0	0	0	0	0	18	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	37	0	25	0	10	0	32	1	187	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CGAC	0	0	0	0	0	0	0	0	5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	22	0	22	0	19	0	50	0	187	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TAGT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	310	0	21	0	22	0	1	0	11	0	0	0	7	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0
ACTA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	310	0	2	0	12	0	0	0	17	0	0	0	11	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0
TAGC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	259	0	0	0	23	0	0	0	24	0	0	0	23	2	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GCTA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	21	0	259	0	0	0	27	0	0	0	35	0	0	0	22	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CAGT	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	12	0	0	0	292	0	29	0	5	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ACTG	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	22	0	0	0	292	0	29	0	6	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CAGC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	27	0	29	0	173	0	0	0	12	0	0	0	9	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GCTG	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	23	0	29	0	173	0	0	0	17	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AAGT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	17	0	0	0	6	0	0	0	329	0	30	0	9	0	0	0	0	0	4	0	11	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ACTT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	11	0	0	0	5	0	0	0	329	0	19	0	22	0	0	0	0	0	1	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AAGC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	35	0	0	0	17	0	19	0	239	0	0	0	22	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GCTT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	24	0	0	0	12	0	30	0	239	0	1	0	38	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GAGT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	11	0	0	0	4	0	0	0	22	0	1	0	249	0	41	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ACTC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	7	0	0	0	1	0	0	0	9	0	0	0	249	0	24	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GAGC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	22	0	0	0	6	0	0	0	38	0	24	0	235	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GCTC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	23	0	0	0	9	0	0	0	22	0	41	0	235	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GGTA	0	0	0	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	232	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TACC	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	232	0	0	0	0	0	0	0	0	0	5	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AAAT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	314	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ATTT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	12	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	314	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AAGA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	4	0	0	0	0	0	0	0	0	0	378	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0
TCTT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	11	0	3	0	0	0	0	0	0	0	0	0	378	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0
AGGA	0	0	11	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	388	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	1
TCCT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	388	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0
CCCC	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	134	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GGGG	0	0	0	0	0	0	0	0	0	0	0	0	11	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	134	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GTTA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	16	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	5	0	0	0	0	0	0	0	0	0	233	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TAAC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	233	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ACCA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	230	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TGGT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	7	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	230	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	8	0
CTAA	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	249	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TTAG	0	0	0	0	0	0	0	0	0	1	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	249	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GTGA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	1	0	0	0	0	0	218	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TCAC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	218	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AATC	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	238	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0
GATT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	238	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	9	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CATA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	237	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TATG	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	237	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TACA	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	259	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TGTA	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	259	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
ACGA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	1	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	353	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	9	0	0	0	0	0	0	0	0
TCGT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	353	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0
ATAA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	242	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TTAT	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	242	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TAAA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	177	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TTTA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	177	0	0	0	0	0	0	0	1	0	2	0	0	0	0	0	0	0	0	0	0	0
TCAA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	241	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TTGA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	241	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0
AAAG	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	292	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
CTTT	1	0	0	0	0	0	0	0	0	0	0	10	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	292	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
AATA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	9	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	272	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TATT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	272	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
TAGA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	299	0	0	0	0	0	0	0	0	0	0	0	0
TCTA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	299	0	0	0	0	0	0	0	0	0	0	0	0	0
GAAA	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	351	0	0	0	0	0	0	0	0	0	0
TTTC	0	0	0	0	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	351	0	0	0	0	0	0	0	0	0	0	0
CCGA	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	310	0	1	0	0	0	0	0	0
TCGG	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	9	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	310	2	4	0	1	0	0	0	0	0
CGGA	0	0	0	0	0	0	0	0	24	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	367	0	0	0	0	0	0
TCCG	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	367	0	0	0	0	0	0	0
CCAG	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	263	9	0	0	0
CTGG	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	263	0	0	3	0	0
ATGG	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	5	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	9	0	0	252	0	0
CCAT	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	3	252	0	0	0
AGCA	0	0	4	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	10	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	8	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	239
TGCT	1	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	6	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	239	1

"""
#PASTE YOUR TABLE HERE (header row + all data rows)
# Example:
# RAW_TABLE = r"""
#     AGTG    CACT    TTCT ...
# AGTG 0       191     0   ...
# CACT 191     1       0   ...
# ...
# """

# Output files
OUT_TSV = Path("neb_getset_matrix.tsv")
OUT_CSV = Path("neb_getset_matrix.csv")


# ============================================================
# PART 2 — PARSER (robust to tabs/spaces)
# ============================================================

def _split_fields(line: str) -> List[str]:
    """Split on tabs or runs of spaces."""
    line = line.strip()
    if not line:
        return []
    # Prefer tabs if present, otherwise split on >=1 whitespace
    if "\t" in line:
        return [x.strip() for x in line.split("\t") if x.strip() != ""]
    return [x.strip() for x in re.split(r"\s+", line) if x.strip() != ""]

def parse_neb_getset_table(raw: str) -> pd.DataFrame:
    """
    Parse the pasted NEB GetSet-like table into a numeric square DataFrame.
    Assumptions:
      - first non-empty line is header: overhangs (columns)
      - each subsequent non-empty line begins with the row overhang, followed by numbers
    """
    lines = [ln for ln in raw.splitlines() if ln.strip() != ""]
    if not lines:
        raise ValueError("RAW_TABLE is empty. Paste the table into RAW_TABLE first.")

    # Find header line: first line that contains at least 2 valid 4-mers
    def is_4mer(x: str) -> bool:
        x = x.upper()
        return len(x) == 4 and all(c in "ACGT" for c in x)

    header_idx = None
    header_cols: List[str] = []
    for i, ln in enumerate(lines):
        fields = _split_fields(ln)
        mers = [f.upper() for f in fields if is_4mer(f)]
        if len(mers) >= 2:
            header_idx = i
            # Use the full list of fields that are 4-mers as column names (in order)
            header_cols = [f.upper() for f in fields if is_4mer(f)]
            break

    if header_idx is None or not header_cols:
        raise ValueError("Could not find a header row with 4-mer column names.")

    # Parse data rows after header
    rows = []
    row_names = []
    for ln in lines[header_idx + 1:]:
        fields = _split_fields(ln)
        if not fields:
            continue
        row_oh = fields[0].upper()
        if not is_4mer(row_oh):
            # skip any non-data line
            continue
        nums = fields[1:]

        # If row is shorter/longer, handle gracefully
        if len(nums) < len(header_cols):
            nums = nums + ["0"] * (len(header_cols) - len(nums))
        elif len(nums) > len(header_cols):
            nums = nums[:len(header_cols)]

        row = pd.to_numeric(pd.Series(nums), errors="coerce").fillna(0.0).astype(float).tolist()
        rows.append(row)
        row_names.append(row_oh)

    if not rows:
        raise ValueError("No data rows parsed. Check RAW_TABLE formatting.")

    df = pd.DataFrame(rows, index=row_names, columns=header_cols)

    # Make square by intersecting index/columns and ordering consistently
    common = [c for c in df.columns if c in df.index]
    df = df.loc[common, common].copy()

    # Ensure uppercase labels
    df.index = df.index.astype(str).str.upper()
    df.columns = pd.Index([str(c).upper() for c in df.columns])

    return df

M = parse_neb_getset_table(RAW_TABLE)

# Save for reuse
M.to_csv(OUT_TSV, sep="\t")
M.to_csv(OUT_CSV)

print(f"Parsed matrix: {M.shape[0]} x {M.shape[1]}")
print(f"Saved: {OUT_TSV.resolve()}")
print(f"Saved: {OUT_CSV.resolve()}")
display(M.head())


# ============================================================
# PART 3 — EXACT MODEL FOR YOUR REGIME (2 inserts into vector; 3 junctions)
# ============================================================

DNA_COMP = str.maketrans({"A": "T", "T": "A", "C": "G", "G": "C"})
def revcomp(s: str) -> str:
    return s.upper().translate(DNA_COMP)[::-1]

def sym_w(M: pd.DataFrame, a: str, b: str, eps: float = 1e-12) -> float:
    """Symmetrized ligation propensity between overhangs a and b."""
    a = str(a).upper().strip()
    b = str(b).upper().strip()
    ra = float(M.at[a, b]) if (a in M.index and b in M.columns) else 0.0
    rb = float(M.at[b, a]) if (b in M.index and a in M.columns) else 0.0
    w = 0.5 * (ra + rb)
    return max(w, eps)

def all_perfect_matchings(nodes: Tuple[int, ...]) -> List[Tuple[Tuple[int, int], ...]]:
    """Enumerate all perfect matchings (pairings) of an even-sized node set."""
    if len(nodes) == 0:
        return [tuple()]
    first = nodes[0]
    out = []
    for k in range(1, len(nodes)):
        partner = nodes[k]
        rest = nodes[1:k] + nodes[k+1:]
        for m in all_perfect_matchings(rest):
            out.append(((first, partner),) + m)
    return out

MATCHINGS_6 = all_perfect_matchings(tuple(range(6)))  # 15 matchings

def intended_probability_for_combo(
    M: pd.DataFrame,
    oh1: str,
    ohmid: str,
    vector_right: str = "AGTG",
    eps: float = 1e-12,
) -> Dict[str, float]:
    """
    Exact probability that intended 3-junction assembly happens (6-end one-shot pairing):

    Ends:
      0 V_L   = oh1
      1 V_R   = vector_right  (fixed AGTG in your case)
      2 I1_L  = revcomp(oh1)
      3 I1_R  = ohmid
      4 I2_L  = revcomp(ohmid)
      5 I2_R  = revcomp(vector_right) = CACT if vector_right=AGTG

    Intended edges: (0,2), (3,4), (1,5)
    """
    oh1 = str(oh1).upper().strip()
    ohmid = str(ohmid).upper().strip()
    vr = str(vector_right).upper().strip()

    ends = (
        oh1,            # 0
        vr,             # 1
        revcomp(oh1),   # 2
        ohmid,          # 3
        revcomp(ohmid), # 4
        revcomp(vr),    # 5
    )

    @lru_cache(maxsize=None)
    def w(i: int, j: int) -> float:
        i, j = (i, j) if i <= j else (j, i)
        return sym_w(M, ends[i], ends[j], eps=eps)

    intended_weight = w(0, 2) * w(3, 4) * w(1, 5)

    total = 0.0
    for matching in MATCHINGS_6:
        prod = 1.0
        for (i, j) in matching:
            prod *= w(i, j)
        total += prod

    p_full = intended_weight / total if total > 0 else 0.0

    def edge_probability(edge: Tuple[int, int]) -> float:
        a, b = edge if edge[0] <= edge[1] else (edge[1], edge[0])
        num = 0.0
        for matching in MATCHINGS_6:
            edges = {tuple(sorted(e)) for e in matching}
            if (a, b) in edges:
                prod = 1.0
                for (i, j) in matching:
                    prod *= w(i, j)
                num += prod
        return num / total if total > 0 else 0.0

    return {
        "p_full_intended": p_full,
        "p_edge_vector_left": edge_probability((0, 2)),
        "p_edge_middle": edge_probability((3, 4)),
        "p_edge_vector_right": edge_probability((1, 5)),
        "oh1": oh1,
        "ohmid": ohmid,
        "vector_right": vr,
        "vector_right_partner": revcomp(vr),
        "oh1_partner": revcomp(oh1),
        "ohmid_partner": revcomp(ohmid),
    }

def rank_overhangs_for_two_insert_vector(
    M: pd.DataFrame,
    vector_overhang1: Sequence[str],
    internal_overhangs: Sequence[str],
    vector_right: str = "AGTG",
    top_n: int = 25,
    forbid_overlap_between_categories: bool = True,
) -> pd.DataFrame:
    """
    Evaluate all (oh1, ohmid) pairs and rank by exact p_full_intended.

    forbid_overlap_between_categories=True is conservative:
      - avoids choosing ohmid that is also in vector_overhang1 (or its RC)
      - avoids choosing oh1 that is also in internal_overhangs (or its RC)
    """
    v1 = [str(x).upper().strip() for x in vector_overhang1]
    mid = [str(x).upper().strip() for x in internal_overhangs]
    vr = str(vector_right).upper().strip()

    v1_set = set(v1)
    v1_rc = {revcomp(x) for x in v1_set}
    mid_set = set(mid)
    mid_rc = {revcomp(x) for x in mid_set}

    rows = []
    for oh1 in v1:
        for ohmid in mid:
            if forbid_overlap_between_categories:
                if (ohmid in v1_set) or (ohmid in v1_rc):
                    continue
                if (oh1 in mid_set) or (oh1 in mid_rc):
                    continue

            # If any ends collapse to fewer than 6 unique sequences, skip (degenerate)
            ends = {oh1, vr, revcomp(oh1), ohmid, revcomp(ohmid), revcomp(vr)}
            if len(ends) < 6:
                continue

            rows.append(intended_probability_for_combo(M, oh1=oh1, ohmid=ohmid, vector_right=vr))

    df = pd.DataFrame(rows).sort_values("p_full_intended", ascending=False).reset_index(drop=True)
    return df.head(top_n)


# ============================================================
# PART 4 — YOUR SETS (as provided)
# ============================================================

VECTOR_OVERHANG1 = [
    "TCTT","TCCT","TCAT","TCGT","AGTT","AGCT",
    "TCTA","TCCA","TCAA","TCGA","AGTA","AGCA",
    "TTCT","TTCC","TTCA","TTCG",
    "CTCT","CTCC","CTCA","CTCG",
    "ATCT","ATCC","ATCA","ATCG",
    "GTCT","GTCC","GTCA","GTCG",
    "TAGT","TAGC","CAGT","CAGC",
    "AAGT","AAGC","GAGT","GAGC"
]
VECTOR_OVERHANG2 = "AGTG"

INTERNAL_OVERHANGS = [
    "CTAA","CAGA","GTGA","CCCC","AATA","ATCA","TACA","ACTC","CGCA","ACCG","ACAT","AAAT",
    "GGTA","TCAA","GAAA","CTGC","GCCC","GACA","ATAC","CATC","CGAG","GAAC","AAAG","CCTA",
    "AAGA","TAAA","ATAA","AGCC","CGAC","TCCA","ACGA","AGAA",
]

# Run ranking
df_top = rank_overhangs_for_two_insert_vector(
    M,
    vector_overhang1=VECTOR_OVERHANG1,
    internal_overhangs=INTERNAL_OVERHANGS,
    vector_right=VECTOR_OVERHANG2,
    top_n=30,
    forbid_overlap_between_categories=True,
)
display(df_top)

# Convenience: show best pick
if len(df_top) > 0:
    best = df_top.iloc[0].to_dict()
    print("\nBEST (by this exact 6-end model):")
    for k in ["oh1","oh1_partner","ohmid","ohmid_partner","vector_right","vector_right_partner",
              "p_full_intended","p_edge_vector_left","p_edge_middle","p_edge_vector_right"]:
        print(f"  {k}: {best[k]}")
